# LoRA: Low-Rank Adaptation - 실습 코드 1: LoRA 구현 및 적용 (PEFT)
### [상세 설명 버전 — 처음 배우는 사람을 위한 확장판]

- Tutorial ID: `expand-lora`
- Tutorial: LoRA: Low-Rank Adaptation
- Section ID: `expand-lora-code-1`
- Section: 실습 코드 1: LoRA 구현 및 적용 (PEFT)

> 이 노트북은 원본 실습 코드(맨 마지막 9번 섹션에 그대로 들어있습니다)를, 처음 LoRA를 공부하는
> 사람도 혼자 힘으로 따라갈 수 있도록 크게 확장한 버전입니다. **수식 → NumPy로 손 계산 →
> 직접 만든 PyTorch 모듈 → 실전 라이브러리(HuggingFace PEFT)** 순서로, "왜 이렇게 하는지"를
> 매 단계마다 설명하면서 진행합니다. 새로운 용어가 나올 때는 바로 다음 문장에서 풀어서
> 설명하려고 했으니, 모르는 단어가 나와도 당황하지 말고 천천히 따라오시면 됩니다.

## 목차
0. LoRA가 왜 필요한가 — 전체 파인튜닝(Full Fine-tuning)의 문제점
1. "저랭크(Low-Rank)"란 무엇인가 (직관 + 아주 작은 예제)
2. LoRA 수식을 한 조각씩 뜯어보기
3. NumPy로 LoRA의 순전파(forward pass)를 손으로 계산해보기
4. 파라미터 개수 비교: Full Fine-tuning vs LoRA
5. NumPy만으로 LoRA의 A, B 행렬을 실제로 "학습"시켜보기 (경사하강법 직접 구현)
6. 학습이 끝난 후 "병합(merge)"이란 무엇인가
7. PyTorch `nn.Module`로 재사용 가능한 LoRA 레이어 만들기
8. 실전 도구: HuggingFace `peft` 라이브러리로 실제 모델(GPT-2)에 LoRA 적용하기
9. (참고) 원본 예제 — Llama-2-7B + QLoRA로 대규모 모델에 적용하는 실전 코드

## 실행 환경 안내
| 섹션 | 필요한 것 | 비고 |
|---|---|---|
| 0 ~ 6 (NumPy 파트) | `numpy` | 어떤 환경에서도 즉시 실행됩니다 |
| 7 (PyTorch 파트) | `torch` (`pip install torch`) | GPU 없이 CPU로도 몇 초 안에 끝납니다 |
| 8 (PEFT 파트) | `transformers`, `peft`, `accelerate` | GPT-2(117M)를 쓰므로 CPU에서도 실행 가능합니다 |
| 9 (원본 예제) | 위 전체 + `bitsandbytes`, GPU, Llama-2 접근 권한 | "실전 규모" 참고 코드입니다 |

각 섹션 코드 셀 위에 무엇이 필요한지 다시 한 번 안내가 붙어 있으니, 지금 당장 설치가
안 되어 있는 부분은 건너뛰고 읽기만 해도 괜찮습니다.

In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: LoRA 구현 및 적용 (PEFT) [상세 설명 버전]
#
# 이 코드는 "정답 코드를 한 번 실행"하는 용도가 아니라,
# LoRA의 수학이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 사전학습 가중치 W0는 그대로 둔 채, 작은 행렬 A, B만으로
#      출력을 바꿀 수 있다는 것을 shape과 실제 숫자로 직접 확인한다.
#   2) B를 0으로 초기화하면 "학습을 시작하는 시점"에는 출력이
#      전혀 바뀌지 않는다는 것을 확인한다.
#   3) 아주 작은 문제에서 A, B를 직접 경사하강법으로 학습시켜
#      손실(loss)이 실제로 줄어드는 것을 눈으로 확인한다.
#   4) 위에서 이해한 원리가 실전 라이브러리(PEFT)의 LoraConfig,
#      target_modules, r, lora_alpha 같은 옵션과 어떻게 연결되는지 확인한다.
#
# 읽는 순서:
#   1) 먼저 0~2번 섹션(마크다운 설명)을 읽고 "왜 저랭크인가"를 이해합니다.
#   2) 3번 섹션에서 아주 작은 숫자(d_in=6, d_out=4, r=2)로 shape이
#      어떻게 이어지는지 눈으로 따라갑니다.
#   3) 5번 섹션에서 실제로 loss가 줄어드는 학습 과정을 관찰합니다.
#   4) 7~8번 섹션에서 같은 원리가 PyTorch, PEFT 코드로 어떻게
#      옮겨지는지 numpy 버전과 비교하며 읽습니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape이 어떻게 변하는지"와
#     "0이었던 값이 학습을 통해 어떻게 바뀌는지"를 눈으로 보세요.
#   - 0~6번은 numpy만 있으면 바로 실행됩니다.
#   - 7번은 torch, 8~9번은 transformers/peft/bitsandbytes 설치와
#     (8~9번은) 인터넷 연결·모델 다운로드가 필요합니다.

## 0. LoRA가 왜 필요한가 — 전체 파인튜닝(Full Fine-tuning)의 문제

거대 언어모델(LLM)을 내 작업에 맞게 "미세조정(fine-tuning)"하고 싶다고 해봅시다.
가장 단순한 방법은 **모델 안의 모든 가중치(weight)를 조금씩 다시 학습시키는 것**입니다.
이것을 **전체 파인튜닝(Full Fine-tuning)** 이라고 부릅니다.

문제는 요즘 LLM은 가중치의 개수(=파라미터 수)가 어마어마하다는 점입니다.
예를 들어 70억 개(7B) 파라미터를 가진 모델이라면:

- **메모리(GPU 메모리) 문제**: 학습 중에는 가중치뿐 아니라, 그래디언트(gradient, 각 가중치를
  어느 방향으로 얼마나 바꿔야 하는지 알려주는 값)와 옵티마이저(optimizer, 가중치를 실제로
  업데이트하는 알고리즘)의 내부 상태까지 함께 저장해야 합니다. 파라미터 1개당 4바이트(32비트)라고
  하면, 가중치만 7B × 4바이트 ≈ 28GB인데, 여기에 그래디언트(약 28GB), 그리고 널리 쓰이는 Adam
  옵티마이저의 보조 값 2종(각각 약 28GB씩)까지 더하면 100GB가 훌쩍 넘는 GPU 메모리가 필요할 수
  있습니다. 일반적인 GPU 한 장(예: 24GB)으로는 어림도 없는 크기입니다.
- **저장 공간 문제**: 작업(task)마다 모델 전체를 다시 저장해야 한다면, 작업이 10개만 있어도
  7B 모델 파일(수십 GB) 10개가 생깁니다.
- **시간 문제**: 파라미터가 많을수록 한 스텝의 계산량도, 전체 학습 시간도 늘어납니다.

**LoRA(Low-Rank Adaptation)** 는 이 문제를 다음과 같은 아이디어로 해결합니다:
"모델 전체를 다시 학습시키지 말고, 원래 가중치는 그대로 얼려(freeze) 두고, 그 옆에 아주 작은
'보정용' 행렬 두 개(A, B)만 새로 학습시키자."

비유를 들면 이렇습니다:

> 두꺼운 원서 교과서(사전학습된 모델, W0)를 통째로 다시 쓰는 대신, 책은 그대로 두고 얇은
> "포스트잇 메모(A, B)"만 필요한 페이지 옆에 붙여서 그 부분만 살짝 다르게 읽히도록 만드는
> 것과 비슷합니다. 책 내용을 지우거나 새로 인쇄할 필요가 없고, 포스트잇만 떼어내면 언제든
> 원래 책으로 돌아갈 수 있습니다. 그리고 포스트잇은 책 전체보다 훨씬 가볍습니다.

여기서 "포스트잇"이 왜 하필 **저랭크(low-rank)** 행렬이어야 하는지, 다음 섹션에서 알아보겠습니다.

## 1. "저랭크(Low-Rank)"란 무엇인가

행렬의 **랭크(rank)** 는 "그 행렬이 몇 개의 독립적인 방향(정보)으로 이루어져 있는가"를 나타내는
값입니다. 말로만 하면 어려우니 예제로 확인해봅시다.

**랭크-1(rank-1) 행렬**: 세로 벡터 하나와 가로 벡터 하나를 곱해서 만든 행렬은, 크기는 커도
"정보량"은 두 벡터를 합친 것만큼밖에 안 됩니다.

```
u = [1, 2, 3]^T   (3×1 세로 벡터)
v = [4, 5]        (1×2 가로 벡터)

u @ v =  [ 4   5 ]
         [ 8  10 ]
         [12  15 ]     (3×2 행렬이지만, 실제로는 u(3개)+v(2개)=5개의 숫자로부터 "만들어진" 것)
```

이 3×2 행렬은 숫자가 6개 들어있지만, 독립적인 정보는 u와 v를 합친 5개(사실 랭크 관점에서는
그보다도 적은 "1"이라는 랭크)뿐입니다. 이런 행렬을 **랭크-1 행렬**이라고 부릅니다.

**랭크-r 행렬**은 이런 랭크-1 행렬을 r개 더한 것과 같고, 이는 곧 (d_out × r) 행렬 B와
(r × d_in) 행렬 A를 곱한 것(`B @ A`)과 같습니다. 즉 두 개의 "작고 마른" 행렬을 곱해서,
"크고 뚱뚱한" (d_out × d_in) 행렬을 만들어낼 수 있습니다. 다만 이렇게 만들어진 행렬의
**랭크는 최대 r을 넘지 못합니다.**

**핵심 포인트 — 숫자를 몇 개나 저장해야 하는가**
- (d_out × d_in) 행렬을 통째로 저장하려면 숫자가 `d_out × d_in`개 필요합니다.
- 반면 "랭크 r짜리"로 표현하면(B와 A만 저장) 숫자가 `r × (d_out + d_in)`개만 있으면 됩니다.
- r이 d_out, d_in보다 훨씬 작다면(`r << d`), 필요한 숫자의 개수가 극적으로 줄어듭니다.

LoRA 논문(Hu et al., 2021)의 핵심 관찰은 다음과 같습니다: "사전학습된 모델을 특정 작업에
맞게 조정할 때 필요한 가중치의 '변화량(ΔW, delta W)'은, 실제로는 랭크가 매우 낮은 경우가
많다." 다시 말해 모델 전체를 다시 학습시키지 않고 "낮은 랭크의 변화량"만 학습해도 충분히
좋은 성능을 얻을 수 있다는 아이디어입니다.

아래 코드로 이 개념을 numpy에서 직접 확인해봅시다.

In [ ]:
import numpy as np

# ------------------------------------------------------------
# (1) 랭크-1 행렬 만들기: 세로 벡터 u와 가로 벡터 v를 곱한다
# ------------------------------------------------------------
u = np.array([[1], [2], [3]])   # shape (3, 1)  -- 세로 벡터
v = np.array([[4, 5]])          # shape (1, 2)  -- 가로 벡터

rank1_matrix = u @ v            # shape (3, 2)
print("랭크-1 행렬:")
print(rank1_matrix)
print("np.linalg.matrix_rank로 확인한 실제 랭크:", np.linalg.matrix_rank(rank1_matrix))
print()

# ------------------------------------------------------------
# (2) 랭크-r 행렬 만들기: (d_out, r) 행렬 B와 (r, d_in) 행렬 A를 곱한다
#     -> 곱해서 나온 행렬은 (d_out, d_in)으로 "크게" 보이지만,
#        실제 랭크는 r을 넘지 않는다.
# ------------------------------------------------------------
d_out, d_in, r = 8, 10, 2
np.random.seed(0)
B_demo = np.random.randn(d_out, r)   # (8, 2)
A_demo = np.random.randn(r, d_in)    # (2, 10)
low_rank_matrix = B_demo @ A_demo    # (8, 10) 이지만...

print("low_rank_matrix shape:", low_rank_matrix.shape)
print("실제 랭크:", np.linalg.matrix_rank(low_rank_matrix), f"  (r={r}을 넘지 않음)")
print()

# ------------------------------------------------------------
# (3) 저장해야 하는 숫자의 개수 비교
# ------------------------------------------------------------
full_numbers = d_out * d_in
low_rank_numbers = r * (d_out + d_in)
print(f"({d_out}x{d_in}) 행렬을 통째로 저장: {full_numbers}개의 숫자")
print(f"B({d_out}x{r}) + A({r}x{d_in})로 저장:  {low_rank_numbers}개의 숫자")
print(f"-> 랭크 r={r}로 표현하면 숫자가 {full_numbers/low_rank_numbers:.1f}배 적게 필요합니다")

## 2. LoRA 수식을 한 조각씩 뜯어보기

이제 위에서 배운 "저랭크 분해"를 실제 신경망의 한 층(layer)에 적용해봅시다.

신경망의 선형 층(Linear layer) 하나는 보통 다음과 같이 계산됩니다:

$$ h = W_0 x $$

- $x$ : 입력 벡터 (예: 토큰 임베딩)
- $W_0$ : 사전학습된 가중치 행렬 (크기: `d_out × d_in`)
- $h$ : 출력 벡터

전체 파인튜닝은 $W_0$ 자체를 $W_0 + \Delta W$ 로 바꿔서 학습합니다. 이때 $\Delta W$도
`d_out × d_in` 크기, 즉 $W_0$와 똑같이 무겁습니다.

**LoRA는 $\Delta W$를 통째로 학습하는 대신, 저랭크로 쪼개서 학습합니다.**

$$ \Delta W = B A $$

- $A$ : 크기 `r × d_in`. 작은 무작위 값으로 초기화합니다.
- $B$ : 크기 `d_out × r`. **0으로 초기화**합니다. (이유는 바로 아래에서 설명합니다.)
- $r$ : 랭크(rank). LoRA 논문에서는 보통 4, 8, 16처럼 아주 작은 값을 사용합니다.

최종 LoRA 순전파(forward) 수식은 다음과 같습니다:

$$ h = W_0 x + \frac{\alpha}{r}\,(BA)\,x $$

- $\alpha$ (alpha) : 스케일링(scaling)을 조절하는 하이퍼파라미터입니다.
- $\frac{\alpha}{r}$ : 실제로 곱해지는 스케일링 계수입니다. r을 바꾸더라도 업데이트의
  "전체적인 크기"가 크게 요동치지 않도록 보정해주는 역할을 합니다.

**왜 하필 B를 0으로 초기화할까요?**
학습을 시작하기 "전" 시점에는 $BA = 0$, 즉 $\Delta W = 0$이 되어야 LoRA를 붙이기 전과
정확히 같은 출력이 나옵니다. 이렇게 하면 학습을 시작하는 바로 그 순간에는 모델 성능이 절대
나빠지지 않는 "안전한 출발점"에서 시작할 수 있습니다. (참고: 반대로 A를 0으로, B를 무작위로
초기화해도 "시작할 때 ΔW=0"이라는 조건은 똑같이 만족하지만, 관례적으로 A는 무작위, B는 0으로
초기화하는 쪽을 사용합니다.)

**학습 중에는 무엇이 바뀔까요?**
- $W_0$ : 그대로 고정(frozen)됩니다. 그래디언트를 계산하지도, 업데이트하지도 않습니다.
- $A$, $B$ : 오직 이 두 행렬만 역전파(backpropagation)를 통해 학습됩니다.

이제 이 수식을 numpy로 한 줄씩 실행해보면서, "말"이 아니라 "숫자"로 확인해보겠습니다.

## 3. NumPy로 LoRA의 순전파를 손으로 계산해보기

지금부터는 아주 작은 숫자를 사용합니다. 실제 LLM에서는 `d_in`, `d_out`이 수천(예: 4096)이고
배치(batch, 한 번에 처리하는 입력 샘플 묶음) 크기도 수십~수백이지만, 여기서는 결과를 눈으로
직접 확인할 수 있도록 아주 작게 잡습니다.

- `batch_size = 3` : 한 번에 넣는 입력 샘플 수 (문장 3개가 들어온다고 생각해도 됩니다)
- `d_in = 6` : 입력 벡터의 차원
- `d_out = 4` : 출력 벡터의 차원
- `r = 2` : LoRA의 저랭크(rank) — 아주 작은 값

아래 코드에서 각 배열의 **shape(모양)** 을 계속 출력해서, "어떤 크기의 배열이 어떤 크기의
배열과 곱해져서 어떤 크기가 나오는지"를 눈으로 따라갈 수 있게 했습니다.

In [ ]:
import numpy as np

# 재현성을 위해 랜덤 시드를 고정합니다 (매번 같은 난수가 나오도록)
np.random.seed(42)

# ------------------------------------------------------------
# 하이퍼파라미터 / 차원 설정
# ------------------------------------------------------------
batch_size = 3   # 한 번에 넣는 입력 샘플 수
d_in = 6         # 입력 벡터의 차원 (예: 토큰 임베딩 차원)
d_out = 4        # 출력 벡터의 차원
r = 2            # LoRA의 저랭크(rank)
alpha = 4        # LoRA 스케일링용 하이퍼파라미터
scaling = alpha / r
print(f"batch_size={batch_size}, d_in={d_in}, d_out={d_out}, r={r}, alpha={alpha}")
print(f"scaling = alpha / r = {alpha} / {r} = {scaling}")
print()

# ------------------------------------------------------------
# "사전학습된(pretrained)" 가중치 W0를 흉내 냅니다.
# 실제로는 며칠~몇 주간 대규모 데이터로 학습되어 만들어진 값이지만,
# 여기서는 랜덤 값으로 대신합니다. shape은 PyTorch의 nn.Linear와
# 동일하게 (d_out, d_in)으로 둡니다. (nn.Linear(in, out).weight의 shape이 (out, in)입니다)
# ------------------------------------------------------------
W0 = np.random.randn(d_out, d_in)
print("W0 (사전학습 가중치) shape:", W0.shape)
print(W0)
print()

# ------------------------------------------------------------
# 입력 X를 만듭니다. shape: (batch_size, d_in)
# ------------------------------------------------------------
X = np.random.randn(batch_size, d_in)
print("X (입력) shape:", X.shape)
print(X)
print()

# ------------------------------------------------------------
# LoRA 없이, 원래 가중치만 사용했을 때의 출력(baseline)을 계산합니다.
# y0 = X @ W0^T   ->  (batch_size, d_in) @ (d_in, d_out) = (batch_size, d_out)
# (W0의 shape이 (d_out, d_in)이므로 W0.T의 shape은 (d_in, d_out)입니다)
# ------------------------------------------------------------
y0 = X @ W0.T
print("y0 (LoRA 적용 전 출력) shape:", y0.shape)
print(y0)

In [ ]:
# ------------------------------------------------------------
# LoRA의 저랭크 행렬 A, B를 만듭니다.
#   - A: (r, d_in)  - 작은 무작위 값으로 초기화
#   - B: (d_out, r) - 0으로 초기화 (2번 섹션에서 설명한 이유!)
# ------------------------------------------------------------
A = np.random.randn(r, d_in) * 0.01   # 아주 작은 무작위 값
B = np.zeros((d_out, r))              # 전부 0

print("A shape:", A.shape)
print(A)
print()
print("B shape:", B.shape, " (값이 전부 0인지 확인)")
print(B)
print()

# ------------------------------------------------------------
# delta_W = B @ A 를 계산합니다.
#   shape: (d_out, r) @ (r, d_in) = (d_out, d_in)  -> W0와 완전히 같은 shape!
#   그래서 나중에 "W_merged = W0 + scaling*delta_W" 로 합칠 수 있습니다. (6번 섹션에서 확인)
# ------------------------------------------------------------
delta_W = B @ A
print("delta_W shape:", delta_W.shape, "  (W0 shape:", W0.shape, "와 동일해야 함)")
print("delta_W 값 (모두 0인지 확인):")
print(delta_W)
print()

# ------------------------------------------------------------
# LoRA가 적용된 최종 출력을 계산합니다.
#   y = X @ W0^T + scaling * (X @ delta_W^T)
# ------------------------------------------------------------
y_lora = X @ W0.T + scaling * (X @ delta_W.T)

print("y_lora shape:", y_lora.shape)
print(y_lora)
print()

# 학습 시작 "전"이므로, B=0 -> delta_W=0 이라서 y0와 y_lora가 완전히 같아야 합니다.
print("y0 와 y_lora 가 같은가? ->", np.allclose(y0, y_lora))
print("(LoRA를 막 붙였을 뿐 아직 학습을 전혀 하지 않았으므로, 원래 모델과 100% 똑같이 동작합니다)")

지금까지는 `B`가 전부 0이라 아무런 변화가 없었습니다. 이번에는 "학습이 어느 정도
진행된 상황"을 흉내 내기 위해, `B`에 손으로 값을 직접 넣어보겠습니다. (실제로는 5번
섹션에서처럼 경사하강법이 이 값을 서서히 채워나가지만, 여기서는 우선 "학습 후에는
이런 느낌"이라는 것을 눈으로 먼저 확인해봅니다.)

In [ ]:
# ------------------------------------------------------------
# "학습이 어느 정도 진행되어 B가 더 이상 0이 아니게 되었다"고 가정하고,
# 무작위 값을 채워 넣어봅니다.
# ------------------------------------------------------------
B_trained = np.random.randn(d_out, r) * 0.1   # "학습 후"라고 가정한 값

delta_W_trained = B_trained @ A
y_lora_trained = X @ W0.T + scaling * (X @ delta_W_trained.T)

print("delta_W_trained shape:", delta_W_trained.shape)
print("delta_W_trained (더 이상 0이 아닙니다):")
print(delta_W_trained)
print()

print("y0             (LoRA 적용 전):")
print(y0)
print()
print("y_lora_trained (학습된 LoRA 적용 후):")
print(y_lora_trained)
print()

diff = y_lora_trained - y0
print("두 출력의 차이 (= LoRA가 얼마나 결과를 바꿔 놓았는지):")
print(diff)
print()
print("y0 와 y_lora_trained 가 다른가? ->", not np.allclose(y0, y_lora_trained))

## 4. 파라미터 개수 비교: Full Fine-tuning vs LoRA

핵심 포인트를 다시 한 번 짚어봅시다. 위 예제에서:

- **전체 파인튜닝**이라면, `W0`와 같은 크기인 `(d_out, d_in)` 행렬 전체(`ΔW`)를 학습해야 합니다.
- **LoRA**는 `A (r, d_in)`와 `B (d_out, r)`, 이 두 개의 작은 행렬만 학습하면 됩니다.

우리가 쓴 아주 작은 예제(d_in=6, d_out=4, r=2)에서는 `r`이 `d_in`, `d_out`에 비해
그리 작지 않아서 절약 효과가 크게 느껴지지 않을 수 있습니다. 이번에는 실제 LLM에
가까운 크기로 바꿔서, LoRA가 얼마나 극적으로 파라미터 수를 줄여주는지 확인해보겠습니다.

In [ ]:
# ------------------------------------------------------------
# (1) 우리가 방금 쓴 작은 예제 기준
# ------------------------------------------------------------
full_finetune_params = W0.size          # d_out * d_in
lora_params = A.size + B.size           # r*d_in + d_out*r

print("### 작은 예제 (d_in=6, d_out=4, r=2) ###")
print(f"Full Fine-tuning 학습 파라미터 수 (W0 전체): {full_finetune_params}")
print(f"LoRA 학습 파라미터 수 (A+B):               {lora_params}")
print(f"LoRA는 전체의 {lora_params/full_finetune_params*100:.1f}% 만 학습합니다")
print()

# ------------------------------------------------------------
# (2) 실제 LLM 은닉층 크기와 비슷한 규모로 계산 (예: hidden size 4096, r=8)
#     -- 층 하나(예: q_proj 하나)만 놓고 비교
# ------------------------------------------------------------
d_in2, d_out2, r2 = 4096, 4096, 8
full2 = d_in2 * d_out2
lora2 = r2 * (d_in2 + d_out2)
print("### 실제 LLM 크기 예시 (하나의 Linear 층, 4096x4096, r=8) ###")
print(f"Full Fine-tuning: {full2:,}개")
print(f"LoRA:             {lora2:,}개")
print(f"비율: {lora2/full2*100:.3f}%  (즉 약 {full2/lora2:.0f}배 적은 파라미터)")
print()

# ------------------------------------------------------------
# (3) Llama-2-7B 전체 모델에 LoRA를 적용한다면? (직접 손으로 추정해보기)
#     - hidden_size=4096, intermediate_size=11008, layer 수=32 (Llama-2-7B 스펙)
#     - target_modules: q_proj, k_proj, v_proj, o_proj (4096->4096) x4
#                        gate_proj, up_proj (4096->11008) x2, down_proj (11008->4096) x1
# ------------------------------------------------------------
hidden = 4096
inter = 11008
n_layers = 32
r3 = 8

per_layer_attn = 4 * r3 * (hidden + hidden)                     # q,k,v,o
per_layer_mlp  = 2 * r3 * (hidden + inter) + r3 * (inter + hidden)  # gate,up,down
per_layer_total = per_layer_attn + per_layer_mlp
total_lora_params = per_layer_total * n_layers

total_model_params = 6_738_415_616   # Llama-2-7B의 공개된 전체 파라미터 수

print("### Llama-2-7B 전체에 LoRA(r=8, 7개 모듈)를 적용한다면? (직접 계산한 추정치) ###")
print(f"레이어 1개당 LoRA 파라미터: {per_layer_total:,}개")
print(f"32개 레이어 전체 LoRA 파라미터: {total_lora_params:,}개")
print(f"전체 모델 파라미터: {total_model_params:,}개")
print(f"비율: 약 {total_lora_params/total_model_params*100:.3f}%")
print()
print("참고: 실제 PEFT 라이브러리의 print_trainable_parameters() 출력값은")
print("      정확한 target_modules 구성, 모델/라이브러리 버전에 따라 위 추정치와")
print("      다소 달라질 수 있습니다. 8~9번 섹션에서 직접 실행해 비교해봅니다.")

## 5. NumPy만으로 A, B를 실제로 "학습"시켜보기

지금까지는 A, B에 숫자를 "손으로" 넣어봤습니다. 이번에는 진짜 학습 알고리즘인
**경사하강법(Gradient Descent)** 을 직접 구현해서, 컴퓨터가 스스로 A, B를 찾아가는 과정을
지켜보겠습니다. (PyTorch 같은 프레임워크를 쓰면 이 과정이 `.backward()` 한 줄로 자동
처리되지만, 여기서는 "그 안에서 실제로 무슨 일이 일어나는지" 직접 확인해봅니다.)

**학습이란 무엇인가 (아주 짧은 복습)**
1. 모델에 입력 `X`를 넣어 예측값 `y_pred`를 만듭니다. (순전파, forward)
2. `y_pred`와 정답 `y_target`이 얼마나 다른지 하나의 숫자(손실, loss)로 계산합니다.
   여기서는 평균제곱오차(MSE, Mean Squared Error)를 사용합니다:
   `loss = mean((y_pred - y_target)^2)`
3. loss를 줄이려면 A, B를 어느 방향으로 움직여야 하는지 계산합니다. (역전파, backward)
   이때 계산되는 "방향"을 그래디언트(gradient, 기울기)라고 부릅니다.
4. A, B를 그래디언트의 반대 방향으로 아주 조금씩(학습률 만큼) 이동시킵니다. (파라미터 업데이트)
5. 1~4를 여러 번(epoch, 전체 데이터를 한 바퀴 도는 횟수) 반복하면 loss가 점점 줄어듭니다.

**합성 문제(synthetic task) 설정**
직접 만든 학습 루프가 잘 동작하는지 확인하려면, "정답을 우리가 이미 알고 있는" 문제를
푸는 것이 가장 확실합니다. 그래서 다음과 같이 진행합니다:

1. 정답이 되는 저랭크 변화량 `A_true`, `B_true`를 미리 랜덤하게 만들어 둡니다.
2. `W_target = W0 + scaling * (B_true @ A_true)` 를 "우리가 도달하고 싶은 이상적인 가중치"로 둡니다.
3. `X`를 랜덤하게 여러 개 뽑고, `y_target = X @ W_target.T` 로 정답 데이터를 만듭니다.
4. 우리 모델(W0는 고정, A/B는 다시 초기값부터 시작)이 경사하강법만으로 이 `(X, y_target)`
   관계를 얼마나 잘 재현해내는지 지켜봅니다.

loss가 0에 가깝게 떨어진다면, "우리가 손으로 유도한 경사하강법 코드가 제대로 동작한다"는
뜻이 됩니다.

In [ ]:
np.random.seed(0)

# ------------------------------------------------------------
# 이번 섹션에서 쓸 차원. (3번 섹션보다 조금 더 키워서 학습 과정이 더 잘 보이게 합니다)
# ------------------------------------------------------------
d_in, d_out, r = 16, 12, 4
alpha = 8
scaling = alpha / r

# "사전학습된" 가중치 (고정, 학습되지 않음)
W0 = np.random.randn(d_out, d_in) * 0.5

# 우리가 "찾아내길 바라는" 정답 저랭크 변화량 (실전에서는 알 수 없지만, 검증을 위해 미리 만들어 둡니다)
A_true = np.random.randn(r, d_in) * 0.5
B_true = np.random.randn(d_out, r) * 0.5
W_target = W0 + scaling * (B_true @ A_true)   # "이상적인, 파인튜닝된" 가중치

# 합성 데이터셋: 입력 X를 랜덤하게 여러 개 뽑고, 정답 W_target으로 y_target을 계산
N = 200   # 전체 데이터 샘플 수
X = np.random.randn(N, d_in)
Y_target = X @ W_target.T   # shape (N, d_out)

print("W0 shape:", W0.shape, " (고정, 학습 안 함)")
print("W_target shape:", W_target.shape, " (우리가 근사하고 싶은 이상적인 가중치)")
print("X shape:", X.shape, "  Y_target shape:", Y_target.shape)
print()

# ------------------------------------------------------------
# 학습을 새로 시작하는 것이므로, A, B는 처음 초기값으로 되돌립니다.
# (A_true, B_true는 "정답 확인용"으로만 쓰고, 학습 루프는 이 값을 전혀 들여다보지 않습니다!)
# ------------------------------------------------------------
A = np.random.randn(r, d_in) * 0.01
B = np.zeros((d_out, r))
print("학습을 시작하는 A, B (A_true, B_true와는 다른, 완전히 새로운 초기값입니다)")

**순전파와 손실 (다시 정리)**

$$ Z = X A^T \qquad (\text{shape: batch} \times r) $$
$$ \hat{y} = X W_0^T + \frac{\alpha}{r}\, Z B^T $$
$$ L = \text{mean}\big((\hat{y} - y_{\text{target}})^2\big) $$

**역전파(그래디언트) — 연쇄법칙(chain rule)을 적용해 미리 유도해 둔 결과**

아래 식들은 위 손실 $L$을 $A$, $B$로 각각 미분해서(연쇄법칙 적용) 얻은 결과입니다. 직접
유도해보고 싶다면 좋은 연습 문제가 되고, 우선은 "이렇게 계산된다"로 받아들이고 코드로
옮겨서 loss가 실제로 줄어드는지부터 확인해도 좋습니다. (19번 셀에서 이 식이 실제로
맞는지 수치적으로 검증도 해봅니다.)

$$ dY = \frac{2}{N}(\hat{y} - y_{\text{target}}) \qquad (N = \text{batch\_size} \times d_{out}, \; \hat y - y_{target}\text{의 전체 원소 개수}) $$
$$ dB = \frac{\alpha}{r}\, dY^T Z $$
$$ dZ = \frac{\alpha}{r}\, dY\, B $$
$$ dA = dZ^T X $$

직관적으로: `dY`는 "예측이 정답보다 얼마나, 어느 방향으로 틀렸는지"이고, 이 오차가
`B`를 거쳐 `dZ`로, 다시 `A`를 거쳐 `dA`로 "거꾸로" 흘러갑니다 (그래서 역전파라고 부릅니다).

이제 이 식을 그대로 코드로 옮겨서, A, B를 반복적으로 업데이트해보겠습니다.

In [ ]:
lr = 0.05          # 학습률 (learning rate) - 한 번에 얼마나 크게 이동할지
epochs = 300        # 전체 반복 횟수
batch_size = 32      # 한 번의 업데이트에 사용할 샘플 수

loss_history = []

for epoch in range(epochs):
    # ---- 미니배치 뽑기 (전체 N개 중 batch_size개를 무작위로) ----
    idx = np.random.choice(N, batch_size, replace=False)
    Xb = X[idx]
    Yb = Y_target[idx]

    # ---- 순전파 (forward) ----
    y0_base = Xb @ W0.T                  # 고정된 W0 경로
    Z = Xb @ A.T                         # (batch, r)
    y_lora_part = scaling * (Z @ B.T)    # (batch, d_out)
    y_pred = y0_base + y_lora_part

    # ---- 손실 계산 ----
    diff = y_pred - Yb
    loss = np.mean(diff ** 2)
    loss_history.append(loss)

    # ---- 역전파 (16번 셀에서 유도한 식을 그대로 코드로 옮김) ----
    Ntot = diff.size
    dY = 2.0 / Ntot * diff
    dZ = scaling * (dY @ B)
    dA = dZ.T @ Xb
    dB = scaling * (dY.T @ Z)

    # ---- 파라미터 업데이트 (경사하강법: 그래디언트의 "반대" 방향으로 이동) ----
    A -= lr * dA
    B -= lr * dB

    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:4d}  loss = {loss:.6f}")

print()
print("학습 시작 loss:", loss_history[0])
print("학습 종료 loss:", loss_history[-1])
print("loss가 충분히 줄었는가? ->", loss_history[-1] < loss_history[0] * 0.01)
print()
print("참고: W0는 이 반복문 안에서 단 한 번도 업데이트되지 않았습니다.")
print("      오직 A, B만 학습되었는데도 손실이 크게 줄었습니다 — 이것이 LoRA의 핵심입니다.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ------------------------------------------------------------
# (환경 설정용 보조 코드) matplotlib의 기본 폰트는 한글 글자를 지원하지
# 않아서, 아래 그래프의 한글 제목이 네모(□□□)로 깨져 보일 수 있습니다.
# 시스템에 설치된 한글 지원 폰트를 자동으로 찾아서 사용하도록 설정합니다.
# (찾지 못하면 그냥 넘어가며, 이 경우 제목만 깨지고 그래프 자체는 정상 출력됩니다)
# ------------------------------------------------------------
_korean_font_candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK JP", "Noto Sans KR"]
_installed_fonts = {f.name for f in fm.fontManager.ttflist}
_found = next((name for name in _korean_font_candidates if name in _installed_fonts), None)
if _found:
    plt.rcParams["font.family"] = _found
else:
    print("[안내] 한글 지원 폰트를 찾지 못했습니다. 그래프의 한글 제목이 깨져 보일 수 있습니다.")
    print("       (Colab이라면 다음을 실행한 뒤 런타임을 재시작하면 해결됩니다)")
    print('       !apt-get -qq install -y fonts-nanum > /dev/null')
plt.rcParams["axes.unicode_minus"] = False   # 마이너스(-) 기호가 깨지는 것을 방지

# ------------------------------------------------------------
# loss가 줄어드는 모습을 그래프로 확인합니다.
# y축을 log scale로 두면, 처음에는 빠르게, 나중에는 서서히 줄어드는
# 전형적인 학습 곡선의 모양이 더 잘 보입니다.
# ------------------------------------------------------------
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("loss (log scale)")
plt.title("NumPy로 직접 구현한 경사하강법으로 LoRA A, B 학습하기")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (선택, 심화) 그래디언트 공식이 정말 맞는지 수치미분으로 검증하기

이 부분은 건너뛰어도 이후 내용을 이해하는 데 지장이 없습니다. 다만 "16번 셀에서 유도한
`dA`, `dB` 식이 정말로 맞는 걸까?"가 궁금하다면, 실무에서 실제로 쓰이는 검증 방법인
**수치 미분(numerical gradient) 검사**를 직접 해볼 수 있습니다.

아이디어는 간단합니다. 미분의 정의를 그대로 이용해서, `A`의 어떤 원소 하나를 아주 살짝
(`+eps`, `-eps`) 흔들어보고 loss가 얼마나 바뀌는지를 재면, 그것이 곧 "그 원소에 대한
그래디언트의 근사값"이 됩니다.

$$ \frac{\partial L}{\partial A_{ij}} \approx \frac{L(A_{ij}+\epsilon) - L(A_{ij}-\epsilon)}{2\epsilon} $$

이 수치적으로 근사한 값과, 우리가 연쇄법칙으로 유도해서 코드로 옮긴 `dA`, `dB` 값이
거의 일치한다면, 우리의 역전파 구현이 올바르다는 강력한 증거가 됩니다.

In [ ]:
np.random.seed(1)

# 검증용으로 새로운 작은 데이터와 A, B를 하나 만듭니다.
A_chk = np.random.randn(r, d_in) * 0.1
B_chk = np.random.randn(d_out, r) * 0.1
Xc = np.random.randn(5, d_in)
Yc = np.random.randn(5, d_out)

def compute_loss(A_, B_):
    # 주어진 A_, B_로 loss를 계산하는 함수 (16번 셀의 순전파 식과 동일)
    y0_ = Xc @ W0.T
    Z_ = Xc @ A_.T
    yl_ = scaling * (Z_ @ B_.T)
    yp_ = y0_ + yl_
    d_ = yp_ - Yc
    return np.mean(d_ ** 2)

# ---- (1) 연쇄법칙으로 유도한 "해석적(analytic)" 그래디언트 계산 ----
y0_ = Xc @ W0.T
Z_ = Xc @ A_chk.T
yl_ = scaling * (Z_ @ B_chk.T)
yp_ = y0_ + yl_
diff_ = yp_ - Yc
Ntot_ = diff_.size
dY_ = 2.0 / Ntot_ * diff_
dZ_ = scaling * (dY_ @ B_chk)
dA_analytic = dZ_.T @ Xc
dB_analytic = scaling * (dY_.T @ Z_)

# ---- (2) 수치미분으로 몇 개의 원소를 직접 흔들어보며 근사 그래디언트 계산 ----
eps = 1e-5

max_err_A = 0.0
for _ in range(5):
    i, j = np.random.randint(r), np.random.randint(d_in)
    A_plus = A_chk.copy();  A_plus[i, j] += eps
    A_minus = A_chk.copy(); A_minus[i, j] -= eps
    numeric_grad = (compute_loss(A_plus, B_chk) - compute_loss(A_minus, B_chk)) / (2 * eps)
    err = abs(numeric_grad - dA_analytic[i, j])
    max_err_A = max(max_err_A, err)

max_err_B = 0.0
for _ in range(5):
    i, j = np.random.randint(d_out), np.random.randint(r)
    B_plus = B_chk.copy();  B_plus[i, j] += eps
    B_minus = B_chk.copy(); B_minus[i, j] -= eps
    numeric_grad = (compute_loss(A_chk, B_plus) - compute_loss(A_chk, B_minus)) / (2 * eps)
    err = abs(numeric_grad - dB_analytic[i, j])
    max_err_B = max(max_err_B, err)

print("A에 대한 (수치미분 vs 연쇄법칙) 최대 오차:", max_err_A)
print("B에 대한 (수치미분 vs 연쇄법칙) 최대 오차:", max_err_B)
print()
print("오차가 0에 아주 가깝다면 (보통 1e-8 이하), 16번 셀의 그래디언트 식이 올바르다는 뜻입니다.")

## 6. 학습이 끝난 후 "병합(Merge)"이란 무엇인가

학습이 끝나 A, B가 좋은 값을 갖게 되었다고 합시다. 실제 서비스(추론, inference)에서는
매번 "① $W_0 x$ 계산 → ② $\frac{\alpha}{r}BAx$ 계산 → ③ 둘을 더하기" 세 단계를 거칠
필요가 없습니다.

$$ W_0 x + \frac{\alpha}{r}(BA) x \;=\; \Big(W_0 + \frac{\alpha}{r}BA\Big)\, x \;=\; W_{\text{merged}}\, x $$

즉 $W_0 + \frac{\alpha}{r}BA$ 를 미리 딱 한 번 계산해서 **하나의 평범한 가중치 행렬**로
만들어버리면(이것을 "병합(merge)"이라고 부릅니다), 추론할 때는 LoRA가 아예 없는 것처럼
행렬곱을 딱 한 번만 하면 됩니다. 즉 **추론 속도에는 전혀 손해가 없습니다.**

(단점도 있습니다: 한 번 병합하면 $A$, $B$가 $W_0$ 안으로 섞여 들어가므로, 이후에
"LoRA만 다시 떼어내기"는 더 이상 할 수 없습니다. 여러 작업(task)마다 다른 LoRA를 갈아
끼우고 싶다면, 병합하지 않고 $A$, $B$를 따로 보관해 두는 것이 유리합니다.)

In [ ]:
# 17번 셀에서 학습이 끝난 A, B를 그대로 사용합니다.

# ---- 병합 전: 지금까지처럼 세 단계로 계산 ----
y_before_merge = X[:5] @ W0.T + scaling * (X[:5] @ (B @ A).T)

# ---- 병합: W0와 delta_W를 미리 하나로 합쳐서 W_merged를 만든다 ----
W_merged = W0 + scaling * (B @ A)
print("W_merged shape:", W_merged.shape, " (W0와 완전히 같은 shape - 평범한 가중치 행렬입니다)")

# ---- 병합 후: 딱 한 번의 행렬곱만으로 계산 ----
y_after_merge = X[:5] @ W_merged.T

print()
print("병합 전 출력과 병합 후 출력이 같은가? ->", np.allclose(y_before_merge, y_after_merge))
print("(수학적으로 당연히 같아야 하고, 실제로도 같습니다 - '병합'은 계산 순서만 바꾼 것입니다)")

## 잠깐 정리

지금까지 numpy만으로 확인한 것을 정리하면 다음과 같습니다.

- ✅ $W_0$는 건드리지 않고, 작은 $A$, $B$ 두 행렬만으로 출력을 원하는 방향으로 바꿀 수 있다
- ✅ $B$를 0으로 초기화하면, 학습을 시작하는 시점에는 출력이 전혀 바뀌지 않는다
- ✅ 저랭크 표현은 전체 가중치를 저장하는 것보다 훨씬 적은 숫자만 있으면 된다 (예: 4096x4096 층에서 약 0.4%)
- ✅ 경사하강법으로 $A$, $B$를 직접 학습시키면 loss가 실제로 줄어든다 (직접 유도한 그래디언트 식으로 확인)
- ✅ 학습이 끝나면 $W_0 + \frac{\alpha}{r}BA$ 로 병합해서, 속도 손해 없이 쓸 수 있다

이제 이 원리를 실제로 신경망을 만들 때 널리 쓰는 도구인 **PyTorch**로 옮겨보겠습니다.
직접 그래디언트 공식을 유도하지 않아도, PyTorch의 자동미분(autograd) 기능이 대신
계산해 줍니다.

> ⚠️ 아래 섹션부터는 `torch`가 설치되어 있어야 실제로 실행됩니다. (`pip install torch`)
> 설치되어 있지 않다면, 코드를 눈으로 읽으면서 앞의 numpy 버전과 한 줄씩 비교해보는
> 것만으로도 충분히 이해할 수 있도록 구성했습니다. (실제로 두 코드는 계산 내용이 완전히 같습니다)

## 7. PyTorch `nn.Module`로 재사용 가능한 LoRA 레이어 만들기

방금 numpy로 했던 것과 정확히 같은 계산을, 이번에는 PyTorch의 `nn.Module`로 감싸서
"어디에나 재사용할 수 있는 부품"으로 만들어보겠습니다. 이렇게 만든 `LoRALinear`는 기존에
학습되어 있는 `nn.Linear`를 통째로 감싸서, 원본은 얼리고(freeze) 그 옆에 A, B만 추가하는
방식으로 동작합니다.

아래 클래스의 각 부분이 numpy 버전의 어떤 코드와 대응되는지 주석에 표시해두었습니다.

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    '''
    이미 학습되어 있는 nn.Linear를 감싸서 LoRA를 적용하는 레이어.

    - base       : 원래 학습된 nn.Linear. 이 안의 weight, bias는 학습하지 않습니다(freeze).
    - lora_A     : (r, in_features)  - 작은 무작위 값으로 초기화 (numpy 버전의 A와 동일)
    - lora_B     : (out_features, r) - 0으로 초기화             (numpy 버전의 B와 동일)
    - scaling    : alpha / r         (numpy 버전의 scaling과 동일)
    '''

    def __init__(self, base_linear: nn.Linear, r: int = 4, alpha: int = 8, dropout: float = 0.0):
        super().__init__()
        self.base = base_linear
        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # ---- 원래 가중치는 고정(freeze)합니다 : LoRA의 핵심 원칙, 원본은 건드리지 않는다 ----
        for p in self.base.parameters():
            p.requires_grad = False

        # ---- 저랭크 행렬 A, B를 "학습 가능한 파라미터"로 추가합니다 ----
        # nn.Parameter로 감싸면 PyTorch가 자동으로 이 텐서를 "학습 대상"으로 추적합니다.
        self.lora_A = nn.Parameter(torch.randn(r, self.in_features) * 0.01)  # numpy: A
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))        # numpy: B (0으로 시작!)

        # ---- (선택) 드롭아웃: LoRA 경로의 입력 일부를 학습 중 무작위로 0으로 만들어 과적합을 줄임 ----
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        # base_out : 고정된 W0 경로 (numpy 버전의 y0_base = X @ W0.T 와 동일)
        base_out = self.base(x)

        # lora_out : LoRA 경로. x -> (dropout) -> A -> B
        # numpy 버전의 "X @ A.T @ B.T" 와 정확히 같은 계산입니다.
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T

        # 최종 출력 = base 경로 + scaling이 적용된 LoRA 경로
        return base_out + self.scaling * lora_out

print("LoRALinear 클래스 정의 완료")

In [ ]:
torch.manual_seed(0)

# "이미 학습되어 있는 층"이라고 가정한 평범한 nn.Linear (사실은 랜덤 초기화 상태입니다)
base_linear = nn.Linear(6, 4)   # in_features=6, out_features=4  (numpy 버전의 d_in, d_out과 동일)

# 위에서 만든 LoRALinear로 감싸기
lora_layer = LoRALinear(base_linear, r=2, alpha=4)

x = torch.randn(3, 6)   # (batch=3, d_in=6)

with torch.no_grad():   # 그래디언트 계산 없이 순전파만 실행
    out_base = base_linear(x)
    out_lora = lora_layer(x)

print("base_linear(x) 출력:")
print(out_base)
print()
print("lora_layer(x) 출력:")
print(out_lora)
print()

# lora_B가 0으로 초기화되어 있으므로, LoRA를 붙이기 전/후 출력이 완전히 같아야 합니다.
print("두 출력이 같은가? ->", torch.allclose(out_base, out_lora))
print("(numpy 버전 9번 셀에서 확인했던 것과 정확히 같은 현상입니다)")

이번엔 실제로 학습까지 시켜보겠습니다. NumPy에서는 16번 셀에서 그래디언트 식을 직접
유도해서 코드로 옮겼지만, PyTorch에서는 `loss.backward()` 한 줄이면 모든 파라미터에 대한
그래디언트가 자동으로 계산됩니다 (이 기능을 **자동미분, autograd** 라고 부릅니다). 우리가
`requires_grad = False`로 얼려둔 `base` 파라미터에는 그래디언트가 계산되지 않고, `lora_A`,
`lora_B`에만 그래디언트가 계산됩니다.

In [ ]:
torch.manual_seed(0)

# ------------------------------------------------------------
# 15번 셀(numpy)과 같은 구조의 합성 문제를 PyTorch로 다시 만듭니다.
# ------------------------------------------------------------
d_in, d_out, r = 16, 12, 4

base_linear = nn.Linear(d_in, d_out)
lora_layer = LoRALinear(base_linear, r=r, alpha=8)

# "이상적으로 도달하고 싶은" 목표 가중치를 하나 랜덤하게 만들어 둡니다.
with torch.no_grad():
    W_target = base_linear.weight.clone() + 0.5 * torch.randn(d_out, d_in)

X = torch.randn(200, d_in)
Y = X @ W_target.T

# ------------------------------------------------------------
# 옵티마이저에는 "학습 가능한(requires_grad=True)" 파라미터만 넘겨줍니다.
# base_linear의 weight, bias는 requires_grad=False이므로 여기 포함되지 않습니다.
# ------------------------------------------------------------
trainable_params = [p for p in lora_layer.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=0.01)
loss_fn = nn.MSELoss()   # 평균제곱오차 (numpy 버전에서 손으로 계산했던 것과 동일한 식)

torch_loss_history = []
for epoch in range(300):
    idx = torch.randperm(200)[:32]     # 미니배치 무작위 추출
    xb, yb = X[idx], Y[idx]

    pred = lora_layer(xb)              # 순전파
    loss = loss_fn(pred, yb)           # 손실 계산
    torch_loss_history.append(loss.item())

    optimizer.zero_grad()              # 이전 스텝의 그래디언트 초기화
    loss.backward()                    # 역전파: lora_A, lora_B의 그래디언트를 자동 계산
    optimizer.step()                   # 그래디언트 반대 방향으로 파라미터 업데이트

    if epoch % 50 == 0 or epoch == 299:
        print(f"epoch {epoch:4d}  loss = {loss.item():.6f}")

print()
print("시작 loss:", torch_loss_history[0])
print("종료 loss:", torch_loss_history[-1])

### 정말로 base(W0)는 학습되지 않았을까? — `requires_grad`로 확인하는 법

PyTorch의 모든 파라미터(`nn.Parameter`)는 `requires_grad`라는 속성을 가지고 있습니다.
- `requires_grad = True` : 이 파라미터는 역전파 시 그래디언트가 계산되고, 옵티마이저가 업데이트합니다.
- `requires_grad = False` : 그래디언트가 계산되지 않고, 값이 절대 바뀌지 않습니다. (= freeze)

`LoRALinear.__init__` 안에서 `self.base`의 모든 파라미터에 `requires_grad = False`를
직접 설정했으므로, 아무리 학습을 반복해도 `base.weight`, `base.bias`는 처음 값 그대로여야
합니다. 아래 코드로 직접 확인해봅니다.

In [ ]:
def count_params(module, only_trainable=False):
    '''모듈 안의 전체(또는 학습 가능한) 파라미터 개수를 세는 헬퍼 함수'''
    if only_trainable:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    return sum(p.numel() for p in module.parameters())

total = count_params(lora_layer)
trainable = count_params(lora_layer, only_trainable=True)

print(f"전체 파라미터 수:        {total}")
print(f"학습 가능한 파라미터 수: {trainable}  (= lora_A + lora_B)")
print(f"비율: {trainable/total*100:.1f}%")
print()

# 각 파라미터 이름과 requires_grad 값을 하나씩 출력해서 직접 눈으로 확인
for name, param in lora_layer.named_parameters():
    print(f"{name:20s}  shape={str(tuple(param.shape)):12s}  requires_grad={param.requires_grad}")

print()
print("base.weight / base.bias 의 requires_grad가 False로 나온다면, 학습 내내 얼어있었다는 뜻입니다.")
print("이것이 바로 8번 섹션에서 볼 HuggingFace PEFT의 model.print_trainable_parameters()가")
print("내부적으로 하는 일과 정확히 같습니다.")

## 8. 실전 도구: HuggingFace `peft` 라이브러리로 실제 모델에 LoRA 적용하기

지금까지는 우리가 직접 `LoRALinear`를 만들었습니다. 하지만 실제 LLM(GPT, Llama, Qwen 등)에는
Q/K/V/출력 projection, MLP 내부의 여러 층 등, LoRA를 적용하고 싶은 위치가 수십~수백 개나
있습니다. 이 모든 곳에 일일이 `LoRALinear`를 만들어 갈아 끼우는 건 번거롭고 실수하기도
쉽습니다.

이런 반복 작업을 자동화해주는 것이 HuggingFace의 **`peft`(Parameter-Efficient Fine-Tuning)**
라이브러리입니다. "이 모델에서, 이런 이름을 가진 층들에, 이런 설정으로 LoRA를 붙여줘"라고
설정(config) 객체 하나만 넘기면, 라이브러리가 모델 내부를 뒤져서 해당 층들을 찾아 지금까지
우리가 만든 것과 같은 방식(원본은 freeze, 그 옆에 작은 A/B 추가)으로 자동으로 감싸줍니다.

> ⚠️ 이번 섹션은 아래 설치가 필요합니다:
> ```bash
> pip install -q transformers peft accelerate
> ```
>
> 원본 실습 코드는 Llama-2-7B(70억 파라미터, 접근 권한 필요)를 4비트로 불러오는 "실전 규모"
> 예제였습니다. 이 노트북에서는 먼저 **GPT-2(1억 2,400만 파라미터, 별도 접근 권한 불필요)** 로
> 같은 과정을 전부 따라가 본 뒤, 맨 마지막(9번 섹션)에서 원본과 같은 Llama-2-7B 예제를
> "실전 규모 참고 코드"로 최신 문법에 맞게 정리해 제공합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"   # 1억 2,400만(124M) 파라미터의 작은 causal LM. 별도 접근 권한이 필요 없습니다.

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ------------------------------------------------------------
# LoraConfig의 target_modules에 뭘 적어야 할지 모르겠다면?
# 모델 구조를 직접 들여다보면 됩니다. LoRA는 보통 nn.Linear(또는 이에 준하는) 층에
# 적용하므로, attention/mlp 관련 하위 모듈 이름을 먼저 살펴봅니다.
# ------------------------------------------------------------
print("=== 모델 전체 파라미터 수 ===")
print(f"{sum(p.numel() for p in model.parameters()):,}")
print()

print("=== 어텐션 관련 하위 모듈 이름 (앞부분 6개만 출력) ===")
count = 0
for name, module in model.named_modules():
    if "attn" in name and type(module).__name__ not in ("GPT2Attention", "ModuleList"):
        print(f"{name:35s} -> {type(module).__name__}")
        count += 1
        if count >= 6:
            break

위 코드를 실행하면 `transformer.h.0.attn.c_attn`, `transformer.h.0.attn.c_proj` 처럼
생긴 이름들이 `Conv1D` 타입으로 출력됩니다 (레이어 번호 `0`부터 GPT-2의 전체 층 수만큼
반복됩니다). 여기서 두 가지를 짚고 넘어갑니다.

1. **GPT-2는 Q, K, V가 하나의 층(`c_attn`)에 합쳐져 있습니다.** Llama류 모델처럼
   `q_proj`, `k_proj`, `v_proj`가 따로 나뉘어 있지 않고, `c_attn` 하나가 세 가지 역할을
   동시에 담당합니다 (출력 차원이 `3 × hidden_size`인 이유입니다).
2. **`nn.Linear`가 아니라 `Conv1D`라는 클래스를 씁니다.** 이름은 "Conv"이지만 실제로는
   완전연결층과 거의 같은 계산을 하며, 내부적으로 가중치가 저장되는 방향(shape)만 조금
   다를 뿐입니다. `peft` 라이브러리가 이 차이를 알아서 처리해주므로, 우리는 그냥
   층의 "이름"만 알려주면 됩니다.

반면 원본 예제(9번 섹션)에서 다루는 Llama류 모델은 `q_proj`, `k_proj`, `v_proj`,
`o_proj`처럼 역할별로 따로 나뉘어 있고 평범한 `nn.Linear`를 씁니다. **모델마다 내부
층의 이름과 구조가 다르므로, 새로운 모델에 LoRA를 적용할 때는 항상 위 코드처럼 구조를
먼저 확인하는 습관을 들이는 것이 좋습니다.** (target_modules에 존재하지 않는 이름을
적으면 `peft`가 오류를 냅니다.)

GPT-2에서는 관례적으로 `target_modules=["c_attn"]` (어텐션만) 또는
`["c_attn", "c_proj"]` (어텐션 + 출력 projection까지)를 많이 사용합니다. 여기서는
가장 간단한 `["c_attn"]`만 사용해보겠습니다.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # 이 모델을 "다음 토큰을 예측하는 언어모델"로 다루겠다는 의미.
                                     # (다른 옵션: SEQ_CLS(분류), TOKEN_CLS, SEQ_2_SEQ_LM 등)

    r=8,                  # 저랭크의 랭크(rank, r). 작을수록 파라미터가 적고 표현력도 제한됨.
                           # 보통 4~64 사이 값을 많이 사용합니다. 여기서는 8.

    lora_alpha=16,         # 스케일링에 쓰이는 alpha. 실제로 곱해지는 값은 scaling = alpha/r = 16/8 = 2.

    lora_dropout=0.05,     # LoRA 경로(A로 들어가기 전)에 5% 확률로 입력 일부를 0으로 만드는 드롭아웃.
                           # 과적합(overfitting, 학습 데이터에만 지나치게 맞춰지는 현상)을 줄이는 정규화 기법.

    bias="none",           # bias(편향) 파라미터는 학습하지 않겠다는 의미.
                           # ("none" | "all" | "lora_only" 중 선택 가능)

    target_modules=["c_attn"],   # 33번 셀에서 확인한, GPT-2의 Q/K/V 통합 어텐션 층 이름.
                                   # 여기 적힌 이름을 가진 층만 골라서 LoRALinear처럼 감쌉니다.
)

print(lora_config)

In [ ]:
lora_model = get_peft_model(model, lora_config)

lora_model.print_trainable_parameters()
# 출력 형태 예시 (실제 숫자는 peft/transformers 버전에 따라 조금 달라질 수 있습니다):
#   trainable params: XXX,XXX || all params: 124,XXX,XXX || trainable%: 0.2X
#
# 4번 섹션에서 우리가 직접 세워본 공식 "r * (in_features + out_features), 층마다 합산"을
# c_attn(768 -> 2304, 12개 층, r=8)에 그대로 적용해보면:
#   r * (768 + 2304) * 12층 = 8 * 3072 * 12 = 294,912개
# 와 비슷한 규모가 나오는지 위 실제 출력과 비교해보세요.

In [ ]:
# ------------------------------------------------------------
# 그래디언트가 실제로 LoRA 파라미터에만 흐르는지 직접 확인해보기
# (30번 셀에서 우리가 직접 만든 LoRALinear로 했던 것과 같은 확인입니다)
# ------------------------------------------------------------
text = "LoRA는 효율적인 파인튜닝 방법입니다."
inputs = tokenizer(text, return_tensors="pt")

# labels를 입력과 동일하게 주면, "다음 토큰 맞히기" 손실을 모델이 자동으로 계산해 줍니다.
outputs = lora_model(**inputs, labels=inputs["input_ids"])
loss = outputs.loss
print("loss:", loss.item())

loss.backward()   # 역전파 실행

print()
print("=== 파라미터별 그래디언트 존재 여부 (일부만 출력) ===")
shown = 0
for name, param in lora_model.named_parameters():
    if param.requires_grad:
        print(f"[학습 대상] {name:55s} grad 계산됨: {param.grad is not None}")
        shown += 1
        if shown >= 3:
            break

frozen_example = [n for n, p in lora_model.named_parameters() if not p.requires_grad][0]
frozen_param = dict(lora_model.named_parameters())[frozen_example]
print(f"[고정(freeze)] {frozen_example:55s} grad 계산됨: {frozen_param.grad is not None}")

### 저장, 다시 불러오기, 병합

학습이 끝나면 세 가지를 할 수 있습니다.

1. **저장**: `save_pretrained()`로 LoRA 가중치(A, B에 해당하는 값들)만 저장합니다.
   원본 GPT-2 가중치는 전혀 바뀌지 않았으므로 다시 저장할 필요가 없고, 그 결과
   저장되는 파일 크기가 매우 작습니다 (전체 모델이 수백 MB라면, LoRA 어댑터는 보통
   수백 KB ~ 수 MB 수준입니다).
2. **다시 불러오기**: 나중에 원본 GPT-2를 새로 불러온 뒤, `PeftModel.from_pretrained()`로
   저장해 둔 LoRA 가중치를 다시 붙일 수 있습니다.
3. **병합**: 6번 섹션에서 배운 것과 정확히 같은 개념으로, `merge_and_unload()`를 호출하면
   $W_0 + \frac{\alpha}{r}BA$ 를 계산해서 LoRA 구조가 사라진 "평범한" 모델 하나로
   합쳐줍니다.

In [ ]:
import os

# 학습 후 LoRA 가중치만 저장 (수 MB 수준 — 전체 GPT-2 모델을 통째로 다시 저장하지 않습니다!)
lora_model.save_pretrained("./gpt2-lora-weights")

print("저장된 파일 목록과 크기:")
for f in sorted(os.listdir("./gpt2-lora-weights")):
    path = os.path.join("./gpt2-lora-weights", f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f:30s} {size_kb:10.1f} KB")

In [ ]:
from peft import PeftModel

# ------------------------------------------------------------
# 원본 GPT-2를 새로 불러온 뒤, 저장해둔 LoRA 가중치를 다시 붙입니다.
# ------------------------------------------------------------
base_model_reloaded = AutoModelForCausalLM.from_pretrained(model_name)
reloaded = PeftModel.from_pretrained(base_model_reloaded, "./gpt2-lora-weights")
print("LoRA 가중치를 다시 불러온 모델 타입:", type(reloaded).__name__)

# ------------------------------------------------------------
# 추론 속도를 위해 하나의 가중치로 합치고 싶다면 (6번 섹션의 "병합"과 정확히 같은 개념!)
# ------------------------------------------------------------
merged_model = reloaded.merge_and_unload()
print("병합 후 모델 타입:", type(merged_model).__name__)
print("-> 이제 순수 GPT2LMHeadModel입니다. LoRA 구조는 사라지고, 그 효과는 가중치 안에 흡수되었습니다.")

## 9. (참고) 실전 규모 예제 — Llama-2-7B + QLoRA

여기서는 이 튜토리얼 시리즈의 **원본 예제**를 최신 문법으로 다듬고, 한 줄씩 설명을 달아서
"실제 7B급 모델에서는 이렇게 쓴다"는 것을 참고할 수 있도록 정리했습니다. 8번 섹션에서
GPT-2로 했던 것과 흐름은 완전히 동일하며, 모델이 훨씬 크다는 점과 4비트 양자화가
추가된다는 점만 다릅니다.

> ⚠️ 실행 요구사항
> - `meta-llama/Llama-2-7b-hf`는 게이트(gated) 모델이라, HuggingFace 계정으로 Meta의
>   라이선스에 동의하고 접근 승인을 받아야 다운로드할 수 있습니다.
> - **4비트 양자화(quantization)** 란, 원래 16비트/32비트로 저장되던 가중치 숫자 하나하나를
>   훨씬 적은 4비트로 압축해서 저장하는 기법입니다. 메모리는 크게 아낄 수 있지만 약간의
>   정밀도 손실이 있을 수 있습니다. 이를 4비트로 압축된 모델과 LoRA를 결합한 방식을
>   **QLoRA**라고 부릅니다.
> - 4비트 양자화에는 `bitsandbytes` 라이브러리가 필요합니다. 예전에는 사실상 CUDA GPU가
>   필수였지만, 최근 `bitsandbytes`는 CPU/Intel XPU 등도 지원하기 시작했습니다. 다만
>   실제로는 GPU에서 훨씬 빠르고 안정적으로 동작하므로, GPU 환경(예: Colab의 GPU
>   런타임)에서 실행하는 것을 권장합니다.
> - `AutoModelForCausalLM.from_pretrained(..., load_in_4bit=True)` 처럼 `load_in_4bit`를
>   바로 넘기는 방식은 최신 `transformers`에서는 **더 이상 권장되지 않습니다(deprecated)**.
>   대신 아래처럼 `BitsAndBytesConfig` 객체를 만들어 `quantization_config`로 넘겨야 합니다.
>   (원본 노트북은 이전 방식을 사용하고 있었는데, 아래 코드에서는 현재 권장되는 방식으로
>   수정했습니다.)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# ------------------------------------------------------------
# 1) 4비트 양자화 설정
#    - load_in_4bit=True              : 가중치를 4비트로 압축해서 불러온다
#    - bnb_4bit_quant_type="nf4"      : QLoRA 논문이 제안한 4비트 데이터 타입(NF4) 사용
#    - bnb_4bit_use_double_quant=True : 양자화에 쓰인 상수까지 한 번 더 양자화해서 메모리를 더 아낌
#    - bnb_4bit_compute_dtype=...     : 저장은 4비트로 작게 하지만, 실제 행렬곱 "계산"은
#                                        bfloat16으로 수행합니다. (저장은 작게, 계산은 정확하게
#                                        — 이것이 QLoRA의 핵심 트릭입니다)
# ------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# ------------------------------------------------------------
# 2) 기본 모델을 4비트로 로드
#    device_map="auto" : accelerate 라이브러리가 GPU/CPU에 층을 자동으로 나누어 배치합니다.
# ------------------------------------------------------------
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    quantization_config=bnb_config,
    device_map="auto",
)

# ------------------------------------------------------------
# 3) LoRA 설정
#    Llama 계열은 GPT-2와 달리 Q/K/V/출력 projection이 각각 분리되어 있고,
#    이름도 q_proj / k_proj / v_proj / o_proj 입니다. MLP도 gate/up/down 세 개로 나뉩니다.
#    (33번 셀에서 설명한 "모델마다 구조가 다르다"는 점이 여기서 실제로 드러납니다)
# ------------------------------------------------------------
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                      # 저랭크 rank
    lora_alpha=16,            # 스케일링 계수 (alpha/r = 2)
    lora_dropout=0.05,
    target_modules=[          # 적용 대상 — 어텐션 4개 + MLP 3개, 총 7개 종류
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# ------------------------------------------------------------
# 4) LoRA 모델 생성 + 학습 가능한 파라미터 수 확인
# ------------------------------------------------------------
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 4번 섹션에서 같은 공식(r * (in+out), 층마다 합산)으로 직접 손 계산했을 때는
# 약 2,000만 개(전체의 약 0.3%) 정도로 추정되었습니다. 실제 target_modules 조합과
# 라이브러리 버전에 따라 정확한 숫자는 달라질 수 있으니, 이 줄을 직접 실행해서
# 나온 값이 최종 기준입니다.

# ------------------------------------------------------------
# 5) 학습 후 LoRA 가중치만 저장 (전체 7B 모델이 아니라 수십 MB 수준!)
# ------------------------------------------------------------
model.save_pretrained("./lora-weights")

## 마무리 — 다음으로 시도해볼 것들

이 노트북에서 만든 numpy/PyTorch 코드를 가지고 직접 바꿔보면서 감을 잡아보세요.

- **`r`(rank)을 바꿔보기**: 17번 셀에서 `r=1`, `r=4`, `r=32`로 바꿔가며 학습 결과(loss)가
  어떻게 달라지는지 비교해보세요. r이 너무 작으면 복잡한 패턴을 표현하지 못해 loss가
  잘 안 줄어들 수 있습니다.
- **`alpha`를 바꿔보기**: `r`은 고정한 채 `alpha`만 키우거나 줄이면, 업데이트의 "속도/크기"가
  어떻게 달라지는지 관찰해보세요.
- **`target_modules`를 바꿔보기**: 34번 셀의 GPT-2 예제에서 `["c_attn"]` 대신
  `["c_attn", "c_proj"]`로 바꾸면, `print_trainable_parameters()`의 결과가 어떻게
  달라지는지 확인해보세요.
- **다른 모델로 확장해보기**: `AutoModelForCausalLM.from_pretrained("다른모델이름")` 뒤에
  `for name, module in model.named_modules(): print(name)` 를 실행해서 구조를 먼저
  살펴보고, 적절한 `target_modules`를 스스로 찾아보세요.

이 노트북에서 다룬 흐름을 한 문장으로 요약하면: **"큰 행렬(W0)은 그대로 두고, 그 옆에
작은 두 행렬(A, B)만 학습시켜서 원하는 방향으로 출력을 조정한다"** 입니다. NumPy로 손수
계산해본 그 원리가, PyTorch의 `nn.Module`을 거쳐, 결국 HuggingFace `peft`의 `LoraConfig`
한 줄로 이어진다는 것을 기억해두면, 앞으로 어떤 LLM에 LoRA를 적용하더라도 "지금 내부에서
어떤 계산이 일어나고 있는지"를 놓치지 않을 수 있을 것입니다.